In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

df.head()

In [ ]:
# Task 2: Write your code here:
df.info()

In [ ]:
# Task 3: Write your code here:
df.describe()

In [ ]:
# Task 4: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery time Distribution')
plt.xlabel('Traffic_Level')
plt.ylabel('delivery_time')
plt.show()




In [ ]:
# Task 1: Write your code here:
df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
df.isnull().sum()
df.dropna()

In [ ]:
# Task 3: Write your code here:
duplicates = df.duplicated().sum()
print("Number of duplicate rows:", duplicates)
df.drop_duplicates(inplace=True)


In [ ]:
#  Encode categorical variables (One Hot Encoding)

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_cols}")

if categorical_cols:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
    print(f"Columns after encoding: {df.columns.tolist()}")
print(f"Dataset shape after encoding: {df.shape}\n")


In [ ]:
# Separate features and target
X = df.drop('delivery_time', axis=1)
y = df['delivery_time']

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Features scaled using StandardScaler")
print(f"Scaled features shape: {X_scaled.shape}")
print(f"Sample of scaled data:\n{X_scaled.head()}\n")

In [ ]:


# Task 2: Use KFold (not StratifiedKFold - this is regression)
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

# Task 3, 4, 5: Train RandomForest and evaluate with MAE
mae_scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_scaled), 1):
    X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train RandomForest
    model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)

    # Predict and evaluate
    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)

    print(f"Fold {fold} - MAE: {mae:.4f}")

print(f"\nAverage MAE across all folds: {np.mean(mae_scores):.4f}")
print(f"Standard Deviation of MAE: {np.std(mae_scores):.4f}\n")

In [ ]:
# Task 1: Write your code here:
# Train final model on all data for plotting
final_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
final_model.fit(X_scaled, y)

# Task 1: Plot feature importance
print("=" * 80)
print("PART 4: PLOTS")
print("=" * 80)

feature_importance = pd.DataFrame({
    'feature': X_scaled.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'][:10], feature_importance['importance'][:10])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Top 10 Feature Importances')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

y_pred_all = final_model.predict(X_scaled)

plt.figure(figsize=(12, 6))
plt.hist(y, bins=30, alpha=0.5, label='Actual', edgecolor='black')
plt.hist(y_pred_all, bins=30, alpha=0.5, label='Predicted', edgecolor='black')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.title('Actual vs Predicted Delivery Time Distribution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task Bonus: Write your code here: